In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 🧹 Étape 2 : Préparation & Nettoyage des Données (Data Wrangling)

Cette étape correspond au deuxième chapitre du pipeline. L'objectif : **auditer** la qualité des données issues de l'acquisition, puis les **nettoyer** rigoureusement à l'aide de notre module `src/data_clean.py`.

**Donnée d'entrée :** `data/processed/matches_merged.csv` — les 30 583 matchs consolidés à l'étape 1 (résultats + classements FIFA historisés + tirs au but).

### 1. Initialisation et imports

In [2]:
import os
import sys
import pandas as pd
import numpy as np

# Ajout du dossier parent pour importer 'src'
sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Librairies prêtes pour le Wrangling !")

Librairies prêtes pour le Wrangling !


### 2. Chargement du dataset et audit initial

On charge la sortie de l'étape 1 et on inspecte sa qualité : dimensions, types des colonnes, doublons et taux de valeurs manquantes.

In [3]:
# Le « brut » de cette étape = la sortie de l'étape 1 (Acquisition).
raw_data_path = '../data/processed/matches_merged.csv'
df_raw = dc.load_raw_data(raw_data_path)

# Audit de qualité.
print(f"Dimensions : {df_raw.shape[0]} matchs, {df_raw.shape[1]} colonnes")
print(f"Doublons   : {df_raw.duplicated().sum()}")
print("\nValeurs manquantes par colonne :")
print(df_raw.isnull().sum())
print()
df_raw.info()

Dimensions : 30583 matchs, 14 colonnes
Doublons   : 0

Valeurs manquantes par colonne :
date                   0
home_team              0
away_team              0
home_score            72
away_score            72
tournament             0
city                   0
country                0
neutral                0
home_rank           1575
home_points         1572
away_rank           1734
away_points         1728
shootout_winner    30078
dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 30583 entries, 0 to 30582
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             30583 non-null  str    
 1   home_team        30583 non-null  str    
 2   away_team        30583 non-null  str    
 3   home_score       30511 non-null  float64
 4   away_score       30511 non-null  float64
 5   tournament       30583 non-null  str    
 6   city             30583 non-null  str    
 7   country          30583 non-null

### 3. Uniformisation des formats de dates

La colonne `date` est chargée comme du texte. On la convertit en type `datetime` et on trie les matchs par ordre chronologique via `dc.clean_dates` — indispensable pour les analyses temporelles et le futur découpage chronologique du modèle.

In [4]:
# Conversion de 'date' en datetime + tri chronologique des matchs.
df_clean = dc.clean_dates(df_raw, 'date')

print(f"Type de la colonne date : {df_clean['date'].dtype}")
print(f"Période couverte : {df_clean['date'].min().date()} → {df_clean['date'].max().date()}")
df_clean.head()

Type de la colonne date : datetime64[us]
Période couverte : 1993-01-01 → 2026-06-27


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rank,home_points,away_rank,away_points,shootout_winner
0,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True,39.0,34.0,69.0,22.0,NaN
1,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False,55.0,27.0,97.0,11.0,NaN
2,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False,71.0,21.0,161.0,0.0,NaN
3,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True,97.0,11.0,69.0,22.0,NaN
4,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False,55.0,27.0,39.0,34.0,NaN


### 4. Anomalies et matchs non exploitables

Deux contrôles :

- **Validité des scores** — un nombre de buts est un entier positif. `dc.handle_outliers` met à `NaN` toute valeur physiquement impossible (score négatif ou absurde).
- **Matchs non joués** — 72 rencontres de la Coupe du Monde 2026 sont déjà au calendrier mais **n'ont pas encore de score**. Leur résultat étant inconnu, on les retire du jeu d'entraînement, mais on les **conserve à part** : ce sont précisément les matchs que le modèle devra prédire (objectif final du projet).

In [5]:
# 4a. Contrôle de validité : un score de but est un entier >= 0.
df_controle = dc.handle_outliers(df_clean, ['home_score', 'away_score'], 0, 99)
scores_invalides = (df_controle[['home_score', 'away_score']].isna().sum().sum()
                    - df_clean[['home_score', 'away_score']].isna().sum().sum())
print(f"Scores physiquement impossibles détectés : {scores_invalides}")

# 4b. Séparation des matchs non joués (score absent = match futur de la CDM 2026).
matchs_futurs = df_controle[df_controle['home_score'].isna()].copy()
df_joues = df_controle[df_controle['home_score'].notna()].copy()

print(f"Matchs joués (conservés pour l'entraînement) : {len(df_joues)}")
print(f"Matchs futurs (mis de côté pour la prédiction) : {len(matchs_futurs)}")
matchs_futurs[['date', 'home_team', 'away_team', 'tournament']].head()

Scores physiquement impossibles détectés : 0
Matchs joués (conservés pour l'entraînement) : 30511
Matchs futurs (mis de côté pour la prédiction) : 72


,date,home_team,away_team,tournament
30511,2026-06-11,Mexico,South Africa,FIFA World Cup
30512,2026-06-11,South Korea,Czech Republic,FIFA World Cup
30513,2026-06-12,Canada,Bosnia and Herzegovina,FIFA World Cup
30514,2026-06-12,United States,Paraguay,FIFA World Cup
30515,2026-06-13,Qatar,Switzerland,FIFA World Cup


### 5. Imputation des valeurs manquantes

- **Classements FIFA** — environ 5 % des équipes (sélections non membres de la FIFA : Catalogne, Zanzibar…) n'ont pas de classement. On leur attribue un rang fictif « non classé » (**220**, au-delà du dernier rang réel qui est 211) et **0** point.
- **Tirs au but** — une valeur manquante signifie simplement « pas de séance de tirs au but ». On rend cette absence explicite avec la valeur `Aucun`.

In [6]:
# 5a. Classements FIFA absents -> rang fictif "non classé" (220) et 0 point.
df_impute = dc.impute_missing_values(df_joues, ['home_rank', 'away_rank'], 220)
df_impute = dc.impute_missing_values(df_impute, ['home_points', 'away_points'], 0.0)

# 5b. Tirs au but : NaN signifie "pas de séance" -> on l'explicite par 'Aucun'.
df_final = dc.impute_missing_values(df_impute, ['shootout_winner'], 'Aucun')

print("Valeurs manquantes restantes après imputation :")
print(df_final.isnull().sum())

Valeurs manquantes restantes après imputation :
date               0
home_team          0
away_team          0
home_score         0
away_score         0
tournament         0
city               0
country            0
neutral            0
home_rank          0
home_points        0
away_rank          0
away_points        0
shootout_winner    0
dtype: int64


### 6. Sauvegarde des données propres

On enregistre deux fichiers dans `data/processed/` :

- `matches_clean.csv` — les matchs joués, nettoyés, prêts pour l'exploration (étapes 3 et 4) ;
- `matches_a_predire.csv` — les matchs futurs de la CDM 2026, mis de côté pour la prédiction finale.

In [ ]:
# Jeu de données nettoyé, prêt pour l'exploration (étapes 3 et 4).
processed_path = '../data/processed/matches_clean.csv'
df_final.to_csv(processed_path, index=False)
print(f"Données propres      : {processed_path}  ({df_final.shape[0]} matchs)")

# Matchs de la CDM 2026 à prédire (objectif final) — conservés séparément.
futurs_path = '../data/processed/matches_a_predire.csv'
matchs_futurs.to_csv(futurs_path, index=False)
print(f"Matchs à prédire     : {futurs_path}  ({len(matchs_futurs)} matchs)")

💾 Données propres      : ../data/processed/matches_clean.csv  (30511 matchs)
💾 Matchs à prédire     : ../data/processed/matches_a_predire.csv  (72 matchs)
